# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saadar2846/flyrank_ml_internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, subprocess
import pandas as pd, numpy as np

REPO_URL = "https://github.com/saadar2846/flyrank_ml_internship"
REPO_DIR = "flyrank_ml_internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Recreate the w04 baseline exactly, so this notebook is self-contained
df["baseline_score"] = (
    (df["days_since_last_update"] >= 180).astype(int) * 0.5
    + (df["impressions_90d"] >= 500).astype(int) * 0.5
)

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Random Forest classifier**, predicting `is_declining_label`.

For Lane 1 (Ranking Signal Analysis), the model is a diagnostic tool, not the end product — it audits
signals more rigorously than pairwise correlations alone (Week 1 already showed correlations can be
misleadingly weak, e.g. avg_position vs ctr at r = -0.073). Random Forest fits because it captures
interactions between signals (staleness only mattering combined with visibility, as in my baseline
rule) and gives interpretable feature/permutation importances — exactly what a signal audit needs.
The starter pipeline already proved this fits the data: it beat the hand-written rule 3x on
Precision@50 (0.740 vs 0.240). I'm skipping Gradient Boosting — extra complexity isn't earning its
keep when interpretability matters more than squeezing out marginal performance.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split: client-grouped train/test split (GroupShuffleSplit on `client_id`), not a plain random split.**

Pages from the same client can share patterns a model could memorize rather than generalize from. A
random split risks putting some of a client's pages in train and others in test, letting the model
cheat on client-specific quirks instead of learning transferable signal. Grouping by client keeps
every client's pages entirely on one side — the same discipline the real pipeline
(`scripts/03_train_model.py`) uses, and matches the leakage checklist from the lane guide.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count", "sessions_90d", "engagement_rate"]

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
baseline_test = df["baseline_score"].iloc[test_idx]

print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")
print(f"Unique clients — train: {df['client_id'].iloc[train_idx].nunique()}, "
      f"test: {df['client_id'].iloc[test_idx].nunique()}")

Train: 23837 rows | Test: 6163 rows
Unique clients — train: 25, test: 7


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Training the Random Forest on the train split, comparing against my Week-4 baseline — on the exact
same held-out test rows, same metric.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

model = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
model.fit(X_train, y_train)
model_score = model.predict_proba(X_test)[:, 1]

rows = []
for k in (20, 50):
    rows.append({
        "k": k,
        "baseline_precision": precision_at_k(baseline_test.values, y_test.values, k),
        "model_precision": precision_at_k(model_score, y_test.values, k),
    })
print(pd.DataFrame(rows))

print(f"\nBaseline ROC AUC: {roc_auc_score(y_test, baseline_test):.3f}")
print(f"Model ROC AUC:    {roc_auc_score(y_test, model_score):.3f}")
print(f"Baseline Avg Precision: {average_precision_score(y_test, baseline_test):.3f}")
print(f"Model Avg Precision:    {average_precision_score(y_test, model_score):.3f}")

    k  baseline_precision  model_precision
0  20                0.55             0.85
1  50                0.58             0.68

Baseline ROC AUC: 0.498
Model ROC AUC:    0.588
Baseline Avg Precision: 0.510
Model Avg Precision:    0.586


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance

perm = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
perm_importances = pd.Series(perm.importances_mean, index=features).sort_values(ascending=False)
print(perm_importances)

test_review = X_test.copy()
test_review["y_true"] = y_test.values
test_review["model_score"] = model_score
false_positives = test_review[(test_review["model_score"] > 0.7) & (test_review["y_true"] == 0)]
false_positives.head(5)

impressions_90d           0.045546
avg_position              0.017135
ctr                       0.014052
sessions_90d              0.010320
content_age_days          0.008989
engagement_rate          -0.001541
word_count               -0.005630
days_since_last_update   -0.013192
dtype: float64


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,sessions_90d,engagement_rate,y_true,model_score
13,238,103,307,39.8,0.00,1342.0,4,0.00,0,0.740
26,300,13,2426,30.0,0.12,2686.0,9,0.00,0,0.885
78,174,92,59,8.7,0.00,1589.0,3,0.00,0,0.855
126,126,20,82,21.0,0.00,2492.0,2,0.00,0,0.810
135,280,25,19802,20.4,0.32,2844.0,102,12.75,0,0.790


**What the model leans on:** [name the top 2-3 features from perm_importances].

**Where it's wrong:** [describe an actual pattern from false_positives — e.g. high impressions but
recently updated, suggesting the model over-weights visibility without discounting recency].

**Compared to my rule:** [state whether the model's top features match or contradict what my Week-4
rule assumed, and what that implies about the rule's thresholds].

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.